# Overview

This notebook contains the data cleaning code for the **Net Overseas Migration**, **Value of Residential Building Work Done** and **Value of Residential Building Work Commenced** datasets.

Information about the **Net Overseas Migration** dataset:
1. This specific Excel file has multiple files within it (tabs), denoting the different years in the Reference period.
2. These "wafers" (as the ABS calls them) will need to be handled separately.
3. Luckily, each wafer follows an identical format so the same cleaning code can be used for each.
4. The plan will be to eventually combine the data from all wafers into a single DataFrame that contains all the information. This can then be saved to disk and used in another notebook for EDA and visualisation.

Since each wafer follows the same format, the cleaning steps will be the same. So we will figure out the cleaning steps for one of the wafers, and iteratively apply this to all wafers.

In [46]:
import pandas as pd
data = pd.read_excel('../data/net_overseas_migration_location_specific.xlsx', sheet_name=0)

data

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,Net Overseas Migration (1),NaN,NaN,NaN,NaN
1,Reference period by State of residence by Dire...,NaN,NaN,NaN,NaN
2,Counting: Persons,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,Filters:,NaN,NaN,NaN,NaN
5,Default Summation,Persons ((x1)),NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN
7,2006,NaN,NaN,NaN,NaN
8,Direction of migration (2),NaN,Arrival,Departure,Total
9,NaN,State of residence,NaN,NaN,NaN


### Preprocessing steps to take
1. [x] Get rid of the header and footer rows (ABS metadata, not relevant to our analysis)
2. [x] Change the column and indices to what is correct (we need the columns to be the direction of migration and indices to be the states and territories)
3. [x] Get rid of any columns that are not needed. (metadata or Excel workbook formatting)

In [47]:
# we will define a cleaning function which takes in a raw dataframe and returns a cleaned version.
def clean_data(raw_data, print_output=False):
    raw_data_cleaned = raw_data.copy()
    data_year = raw_data_cleaned.iloc[7, 0]
    raw_data_cleaned.columns = raw_data_cleaned.loc[8]
    raw_data_cleaned = raw_data_cleaned.rename_axis('Direction of migration', axis='columns')
    raw_data_cleaned.index = raw_data_cleaned.iloc[:, 1]
    raw_data_cleaned = raw_data_cleaned.iloc[10:25, 2:]
    raw_data_cleaned['Arrival'] = raw_data_cleaned['Arrival'].astype(int)
    raw_data_cleaned['Departure'] = raw_data_cleaned['Departure'].astype(int)
    raw_data_cleaned['Total'] = raw_data_cleaned['Total'].astype(int)
    # this column will be important for when we merge DataFrames
    raw_data_cleaned['Year'] = data_year
    if print_output:
        print('=== DATASET INFORMATION ===')
        print('Dataset Type:', type(raw_data_cleaned))
        print('Data Shape:', raw_data_cleaned.shape)
        print()
        print('Data types present in dataset:')
        print(raw_data_cleaned.dtypes)
        print()
        print(raw_data_cleaned.describe())
    return raw_data_cleaned

### Next step: Combining all cleaned DataFrames into a single DataFrame that contains information across years.

In [48]:
# Sample output of the data from one of the sheets.
sample = clean_data(pd.read_excel('../data/net_overseas_migration_location_specific.xlsx', sheet_name=0))
sample

Direction of migration,Arrival,Departure,Total,Year
nan,,,,
New South Wales,75020,-39100,35920,2006
Victoria,51670,-23810,27860,2006
Queensland,41240,-21250,19980,2006
South Australia,11820,-4640,7180,2006
Western Australia,26430,-11520,14910,2006
Tasmania,1750,-950,800,2006
Northern Territory,2330,-2080,260,2006
Australian Capital Territory,3170,-2500,670,2006
Not Stated,0,0,0,2006


In [49]:
all_frames = []
for i in range(0, 20):
    frame_i = clean_data(pd.read_excel('../data/net_overseas_migration_location_specific.xlsx', sheet_name=i))
    all_frames.append(frame_i)

cleaned_data = pd.concat(all_frames)
cleaned_data['State-Territory'] = cleaned_data.index
cleaned_data.reset_index(drop=True, inplace=True)
cleaned_data

Direction of migration,Arrival,Departure,Total,Year,State-Territory
0,75020,-39100,35920,2006,New South Wales
1,51670,-23810,27860,2006,Victoria
2,41240,-21250,19980,2006,Queensland
3,11820,-4640,7180,2006,South Australia
4,26430,-11520,14910,2006,Western Australia
...,...,...,...,...,...
295,10,-10,10,2025,Christmas Island
296,0,0,0,2025,Jervis Bay
297,0,0,0,2025,Cocos (Keeling) Islands
298,0,0,0,2025,Other Territories


In [50]:
cleaned_data.groupby('Year')['Total'].sum()

Year
2006     215150
2007     488070
2008     631400
2009     493810
2010     344070
2011     412480
2012     480500
2013     416740
2014     364680
2015     373450
2016     487670
2017     483320
2018     504470
2019     495240
2020      -9930
2021      18610
2022     875730
2023    1061230
2024     659900
2025     320320
Name: Total, dtype: int64

In [51]:
cleaned_data.to_csv('../data/migration_data.csv')

# Summary
1. [x] Extracted each sheet from the Excel file.
2. [x] Got rid of all headers and footers, as well as unnecessary columns.
3. [x] Concatenated all the cleaned DataFrames together.
4. [x] Migration data is now ready to be analysed.

## Next: Cleaning **Value of Residential Building Work Done**

In [52]:
work_done_raw = pd.read_excel('../data/value_of_residential_building_work_done.xlsx', sheet_name=1)
work_done_raw

,Unnamed: 0,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; New ; Houses ;,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; New ; Houses ;.1,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; New ; Total Other Residential ;,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; New ; Total Other Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; New ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; Alterations and additions including conversions ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; Alterations and additions including conversions ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; Total (Type of Work) ; Total Residential ;,...,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; New ; Houses ;,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; New ; Houses ;.1,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; New ; Total Other Residential ;,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; New ; Total Other Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; New ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; Alterations and additions including conversions ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; Alterations and additions including conversions ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; Total (Type of Work) ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; Total (Type of Work) ; Total Residential ;.1
0,Unit,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,...,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000
1,Series Type,Original,Seasonally Adjusted,Original,Seasonally Adjusted,Original,Seasonally Adjusted,Original,Seasonally Adjusted,Original,...,Original,Seasonally Adjusted,Original,Seasonally Adjusted,Original,Seasonally Adjusted,Original,Seasonally Adjusted,Original,Seasonally Adjusted
2,Data Type,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,...,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW
3,Frequency,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,...,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter
4,Collection Month,3,3,3,3,3,3,3,3,3,...,3,3,3,3,3,3,3,3,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,2024-09-01 00:00:00,3257180,3180905,3166233,3058919,6423413,6239825,1201286,1143643,7624699,...,141453,139139,328337,315971,469790,455111,36370,34662,506159,489773
210,2024-12-01 00:00:00,3313194,3276988,3140204,3184602,6453399,6461590,1246500,1161789,7699898,...,134643,131678,310184,301694,444827,433372,38898,34570,483726,467942
211,2025-03-01 00:00:00,3021398,3155434,3070441,3226162,6091839,6381595,1078040,1214053,7169879,...,119902,125913,262842,297961,382744,423875,39734,45831,422477,469706
212,2025-06-01 00:00:00,3114839,3093271,3336913,3244094,6451751,6337366,1204814,1211157,7656565,...,125688,124874,347105,332842,472793,457716,43393,43336,516186,501052


In [53]:
work_commenced_raw = pd.read_excel('../data/value_of_residential_building_work_commenced.xlsx', sheet_name=1)
work_commenced_raw.head()

,Unnamed: 0,Value of work commenced ; Chain Volume Measures ; New South Wales ; New ; Houses ;,Value of work commenced ; Chain Volume Measures ; New South Wales ; New ; Total Other Residential ;,Value of work commenced ; Chain Volume Measures ; New South Wales ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; New South Wales ; Alterations and additions including conversions ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Victoria ; New ; Houses ;,Value of work commenced ; Chain Volume Measures ; Victoria ; New ; Total Other Residential ;,Value of work commenced ; Chain Volume Measures ; Victoria ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Victoria ; Alterations and additions including conversions ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Queensland ; New ; Houses ;,...,Value of work commenced ; Chain Volume Measures ; Tasmania ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Tasmania ; Alterations and additions including conversions ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Northern Territory ; New ; Houses ;,Value of work commenced ; Chain Volume Measures ; Northern Territory ; New ; Total Other Residential ;,Value of work commenced ; Chain Volume Measures ; Northern Territory ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Northern Territory ; Alterations and additions including conversions ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Australian Capital Territory ; New ; Houses ;,Value of work commenced ; Chain Volume Measures ; Australian Capital Territory ; New ; Total Other Residential ;,Value of work commenced ; Chain Volume Measures ; Australian Capital Territory ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Australian Capital Territory ; Alterations and additions including conversions ; Total Residential ;
0,Unit,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,...,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000
1,Series Type,Original,Original,Original,Original,Original,Original,Original,Original,Original,...,Original,Original,Original,Original,Original,Original,Original,Original,Original,Original
2,Data Type,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,...,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW
3,Frequency,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,...,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter
4,Collection Month,3,3,3,3,3,3,3,3,3,...,3,3,3,3,3,3,3,3,3,3


Since both datasets have a very similar format, their cleanup will be similar.

### Phase 1: Clean the Value of Work Done dataset

### Steps to cleaning this dataset
- Remove all unnecessary metadata rows (first 9 rows).
- Isolate the **seasonally adjusted** data to look at the underlying trends without the typical seasonal effects such as summer holiday spikes in each year.
- Take the **Total Residential** data columns for each state and save them in a separate DataFrame.
- Make sure both datasets start from the same quarter and end at the same quarter. For maximum accuracy, we will look at data as far back as both go, which is 1980.
- Lastly, we expect there to be 8 columns (6 states, 2 territories of mainland Australia) present in our cleaned DataFrame.

We are interested in getting all the columns that follow the naming template below:


**Value of work done during quarter ; Chain Volume Measures ; <State/Territory> ; New ; Total Residential**

In [54]:
import re
work_done = work_done_raw.copy()
pattern = r'Value of work done during quarter\s*;\s*Chain Volume Measures ;\s*[^;]+\s*;\s*New\s*;\s*Total Residential\s*;'
columns_mask = [re.match(pattern, s) is not None for s in work_done.columns]
work_done.index = work_done.loc[:, 'Unnamed: 0']
work_done = work_done.iloc[33:, columns_mask]
work_done = work_done.rename_axis('Quarter', axis='rows')
work_done

,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; New South Wales ; New ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; Victoria ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; Victoria ; New ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; Queensland ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; Queensland ; New ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; South Australia ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; South Australia ; New ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; Western Australia ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; Western Australia ; New ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; Tasmania ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; Tasmania ; New ; Total Residential ;.1,Value of work done during quarter ; Chain Volume Measures ; Northern Territory ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; New ; Total Residential ;,Value of work done during quarter ; Chain Volume Measures ; Australian Capital Territory ; New ; Total Residential ;.1
Quarter,,,,,,,,,,,,,,,
1980-09-01 00:00:00,4088764,3814964,1989332,1887469,2441909,2288416,447523,429910,1128290,1090740,266348,266840,173020,134753,133871
1980-12-01 00:00:00,4031912,3799237,1828875,1764553,2345220,2231782,429998,410493,1018430,974481,301181,279873,159208,147704,134002
1981-03-01 00:00:00,3575463,3894618,1500960,1667926,2172663,2548562,371163,405729,949397,1022381,218346,240093,139579,160347,172141
1981-06-01 00:00:00,3844366,3882976,1626414,1591926,2551300,2447961,373040,388711,1046267,1049542,228966,226367,146849,170831,176517
1981-09-01 00:00:00,3910040,3649102,1672899,1591675,2953227,2775027,351760,338093,1094516,1058868,239161,239840,170476,159730,158319
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-09-01 00:00:00,6423413,6239825,7170477,6873499,4405496,4109527,1275619,1208571,1987987,1916982,252987,250093,57446,469790,455111
2024-12-01 00:00:00,6453399,6461590,6646086,6892943,4158744,4157601,1321422,1335900,1904090,1933378,263958,263942,61809,444827,433372
2025-03-01 00:00:00,6091839,6381595,6762082,7048166,4037295,4370674,1304479,1395330,1883642,1964409,246029,258923,60786,382744,423875


In [55]:
# final edit: every column that shows seasonalized data must have it specified in their column name
print('==== COLUMN NAMES BEFORE ====')
print(work_done.columns)
work_done.columns = [col.replace('.1', ' (SA)') for col in work_done.columns]
print('==== COLUMN NAMES AFTER ====')
print(work_done.columns)

==== COLUMN NAMES BEFORE ====
Index(['Value of work done during quarter ;  Chain Volume Measures ;  New South Wales ;  New ;  Total Residential ;',
       'Value of work done during quarter ;  Chain Volume Measures ;  New South Wales ;  New ;  Total Residential ;.1',
       'Value of work done during quarter ;  Chain Volume Measures ;  Victoria ;  New ;  Total Residential ;',
       'Value of work done during quarter ;  Chain Volume Measures ;  Victoria ;  New ;  Total Residential ;.1',
       'Value of work done during quarter ;  Chain Volume Measures ;  Queensland ;  New ;  Total Residential ;',
       'Value of work done during quarter ;  Chain Volume Measures ;  Queensland ;  New ;  Total Residential ;.1',
       'Value of work done during quarter ;  Chain Volume Measures ;  South Australia ;  New ;  Total Residential ;',
       'Value of work done during quarter ;  Chain Volume Measures ;  South Australia ;  New ;  Total Residential ;.1',
       'Value of work done during quarter 

In [56]:
work_commenced = work_commenced_raw.copy()
pattern = r'Value of work commenced\s*;\s*Chain Volume Measures ;\s*[^;]+\s*;\s*New\s*;\s*Total Residential\s*;'
columns_mask = [re.match(pattern, s) is not None for s in work_commenced.columns]
work_commenced.index = work_commenced.loc[:, 'Unnamed: 0']
work_commenced = work_commenced.iloc[9:, columns_mask]
work_commenced = work_commenced.rename_axis('Quarter', axis='rows')
work_commenced

,Value of work commenced ; Chain Volume Measures ; New South Wales ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Victoria ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Queensland ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; South Australia ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Western Australia ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Tasmania ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Northern Territory ; New ; Total Residential ;,Value of work commenced ; Chain Volume Measures ; Australian Capital Territory ; New ; Total Residential ;
Quarter,,,,,,,,
1969-09-01 00:00:00,3229627,2000160,1043840,514416,1232832,319555,78450,152902
1969-12-01 00:00:00,3003763,1909886,953878,495823,1082639,189445,67495,150166
1970-03-01 00:00:00,3216247,1794135,1016709,483737,1101781,231402,93292,228897
1970-06-01 00:00:00,2949607,1781120,980653,521854,962632,193683,86577,196371
1970-09-01 00:00:00,2811668,1899363,1069900,572366,921771,265731,119795,229201
...,...,...,...,...,...,...,...,...
2024-09-01 00:00:00,6902098,7068121,4739307,1226155,2259667,285063,35253,137798
2024-12-01 00:00:00,6014272,6632368,4735912,1302920,2133662,255711,60725,369587
2025-03-01 00:00:00,6577509,7575847,3973224,1499104,2215002,245743,81048,200633


# Summary
1. [x] Changed the index of the data to the timestamps for time series analysis.
2. [x] Isolated both the columns for the original and seasonally adjusted data for each of the states and territories using a Regex pattern.
3. [x] Removed all unnecessary rows, including empty ones.

In [57]:
# save both cleaned data into new CSV files
work_done.to_csv('../data/work_done.csv')
work_commenced.to_csv('../data/work_commenced.csv')